<h1 align="center"> Aplicações em Processamento de Linguagem Natural </h1>
<h2 align="center"> Aula 03 - Técnicas de Pré-Processamento de Texto e Similaridade</h2>
<h3 align="center"> Prof. Fernando Vieira da Silva MSc.</h3>

<h2> Técnicas para Pré-Processamento - Parte 2</h2>

<p>Uma vez que o texto já foi devidamente tratado, removendo stopwords e pontuações, e aplicando stemming ou lemmatization, agora precisamos contar a frequência das palavras (ou n-grams) que utilizaremos em seguida como atributos para as técnicas de aprendizado de máquina.</p>

<b>1. TF-IDF (Term Frequency - Inverse Document Frequency)</b>

<p><b>Term Frequency:</b> um termo que aparece muito em um documento, tende a ser um termo importante. Em resumo, divide-se o número de vezes em que um termo apareceu pelo maior número de vezes em que algum outro termo apareceu no documento.</p>

tf<sub>wd</sub> = f<sub>wd</sub> / m<sub>wd</sub>

onde:<br>
f<sub>wd</sub> é o número de vezes em que o termo <i>w</i> aparece no documento <i>d</i>.<br>
m<sub>wd</sub> é o maior valor de f<sub>wd</sub> obtido para algum termo do documento <i>d</i><br>

<p><b>Inverse Document Frequency:</b> um termo que aparece em poucos documentos pode ser um bom descriminante. Obtem-se dividindo o número de documentos pelo número de documentos em que o termo aparece.</p>

idf<sub>w</sub> = log<sub>2</sub>(n / n<sub>w</sub>)

onde:<br>
n é o número de documentos no corpus
n<sub>w</sub> é o número de documentos em que o termo <i>w</i> aparece.

Na prática, usa-se:

tf-idf = tf<sub>wd</sub> * idf<sub>w</sub>

Podemos calcular o TF-IDF de um corpus usando o pacote <b>scikit-learn</b>. Primeiramente, vamos abrir novamente o texto de Hamlet e armazenar as sentenças em uma ndarray do numpy (como se cada sentença fosse um documento do corpus):

In [1]:
import nltk
import numpy as np
from nltk.tokenize import sent_tokenize

hamlet_raw = nltk.corpus.gutenberg.raw('shakespeare-hamlet.txt')

sents = sent_tokenize(hamlet_raw)

hamlet_np = np.array(sents)

print(hamlet_np.shape)


(2355,)


<p>Agora vamos definir uma função para tokenização pelo scikit-learn.</p>

In [2]:
from nltk import pos_tag
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import string
from nltk.corpus import wordnet

stopwords_list = stopwords.words('english')

lemmatizer = WordNetLemmatizer()

def my_tokenizer(doc):
    words = word_tokenize(doc)
    
    pos_tags = pos_tag(words)
    
    non_stopwords = [w for w in pos_tags if not w[0].lower() in stopwords_list]
    
    non_punctuation = [w for w in non_stopwords if not w[0] in string.punctuation]
    
    lemmas = []
    for w in non_punctuation:
        if w[1].startswith('J'):
            pos = wordnet.ADJ
        elif w[1].startswith('V'):
            pos = wordnet.VERB
        elif w[1].startswith('N'):
            pos = wordnet.NOUN
        elif w[1].startswith('R'):
            pos = wordnet.ADV
        else:
            pos = wordnet.NOUN
        
        lemmas.append(lemmatizer.lemmatize(w[0], pos))

    return lemmas
    
    

E essa função será chamada pelo objeto TfidfVectorizer

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

hamlet_raw = nltk.corpus.gutenberg.raw('shakespeare-hamlet.txt')

sents = sent_tokenize(hamlet_raw)

hamlet_np = np.array(sents)

tfidf_vectorizer = TfidfVectorizer(tokenizer=my_tokenizer)

tfs = tfidf_vectorizer.fit_transform(hamlet_np)

print(tfs.shape)

(2355, 4305)


In [4]:
print([k for k in tfidf_vectorizer.vocabulary_.keys()][:20])

['tragedie', 'hamlet', 'william', 'shakespeare', '1599', 'actus', 'primus', 'scoena', 'prima', 'enter', 'barnardo', 'francisco', 'two', 'centinels', "'s", 'fran', 'nay', 'answer', 'stand', 'vnfold']


In [5]:
print(tfs[:50,:50])#linha 0 coluna 17

  (0, 17)	0.4098405429500896
  (4, 5)	1.0
  (12, 6)	0.25569565102708125
  (13, 6)	0.24183188573710981
  (22, 5)	0.5865767987601467
  (27, 0)	0.226773205328385
  (29, 5)	0.28458175550618814
  (37, 5)	0.26402019659267234
  (37, 0)	0.23602778459784524
  (39, 6)	0.13875455587670743
  (40, 13)	0.35297267044370917
  (41, 25)	0.31475626155215114
  (43, 5)	0.14416024946136125
  (46, 5)	0.32299630350190056


<b>2. TF-IDF de N-gramas</b>

Opcionalmente, podemos obter os atributos tf-idf de n-grams, combinando as classes CountVectorizer e TfidfTransformer. Em nosso exemplo, vamos utilizar apenas trigramas:

In [7]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer

count_vect = CountVectorizer(ngram_range=(3,3))

n_gram_counts = count_vect.fit_transform(hamlet_np)

tfidf_transformer = TfidfTransformer()

tfs_ngrams = tfidf_transformer.fit_transform(n_gram_counts)

print(tfs_ngrams.shape)

(2355, 23483)


<b>3. Redução de Dimensionalidade</b>

<p>A transformação do corpus em atributos contendo as frequências TF-IDF em geral resultará numa ndarray bastante esparsa, ou seja, com muitas dimensões. Porém, além de isso tornar o treinamento de algoritmos mais demorado e custoso (computacionalmente falando), muitas dessas dimensões provavelmente são pouco representativas ou mesmo podem causar ruído durante o treinamento. Para resolver esse problema, podemos aplicar uma técnica de redução de dimensionalidade simples chamada <b>Singular Value Decomposition (SVD)</b>. 

<p>Essa técnica transformará os vetores da matriz original, rotacionando e escalando-os, resultando em novas representações. A redução de dimensionalidade é feita ao manter apenas as <i>k</i> dimensões mais representativas que escolhermos. Outra vantagem dessa técnica é que as dimensões originais são, de certa forma, "combinadas", o que resulta em uma nova forma de representar a combinação de termos. No contexto de PLN, essa técnica é conhecida como <b>Latent Semantic Analysis (LSA)</b></p>

In [8]:
from sklearn.decomposition import TruncatedSVD

svd_transformer = TruncatedSVD(n_components=1000)

svd_transformer.fit(tfs)

print(sorted(svd_transformer.explained_variance_ratio_)[::-1][:30])#lista de variância dos componentes, os 30 primeiros

[0.043191439648642097, 0.0186196488852901, 0.01599182682145728, 0.015984996712032976, 0.013744333318750659, 0.011771580238263623, 0.009650213667975397, 0.009516807424958135, 0.009505050204172184, 0.009460542064125707, 0.009317453675259471, 0.009270042565307376, 0.007711406891193048, 0.006983997622721751, 0.006654688885419298, 0.006419528576288445, 0.005885302030095432, 0.005372277900299407, 0.005158171190663278, 0.004931257956478708, 0.0048675271120071535, 0.0047897360657529675, 0.004462492864775194, 0.004339957865186572, 0.004296902243618564, 0.004160561728496293, 0.004039480646198127, 0.00391373336970149, 0.0038585289998033586, 0.003829828436706046]


<p>Agora vamos manter as dimensões até que a variância acumulada seja maior ou igual a 0.50.</p>

In [9]:
cummulative_variance = 0.0
k = 0
for var in sorted(svd_transformer.explained_variance_ratio_)[::-1]:
    cummulative_variance += var
    if cummulative_variance >= 0.5:
        break
    else:
        k += 1
        
print(k)#143 colunas

143


<p>Transformarmos novamente, mas desta vez com o número de k componentes que obtemos anteriormente.</p>

In [10]:
svd_transformer = TruncatedSVD(n_components=k)
svd_data = svd_transformer.fit_transform(tfs)
print(sorted(svd_transformer.explained_variance_ratio_)[::-1])

[0.043191439648500314, 0.018619648886411405, 0.015991826821840686, 0.015984996725227203, 0.01374433329474433, 0.011771580248388456, 0.009650213671718361, 0.009516807654117335, 0.00950505033921406, 0.009460542035365274, 0.009317453709580606, 0.009270042505481989, 0.007711407746961901, 0.006983999173400605, 0.0066546887072167155, 0.006419526780009337, 0.00588530001584942, 0.00537227733978046, 0.005158170260666824, 0.004931258798450128, 0.004867525986863745, 0.00478973701492329, 0.0044624819211732, 0.004339949052106818, 0.00429689953910693, 0.004160563820234372, 0.004039479480255034, 0.0039137255525435325, 0.003858525468255889, 0.003829812564247057, 0.0037136250455372375, 0.003663403454473603, 0.0035824599540086027, 0.003561725944514837, 0.0035351633705145605, 0.0034284504049944264, 0.003345059630482835, 0.0032429602589981693, 0.0032341168288134614, 0.0031762185855394674, 0.003122638915489024, 0.003042235994118686, 0.0029313769783966184, 0.0029183221702665514, 0.0028876372202308203, 0.002

In [11]:
print(svd_data.shape)

(2355, 143)


In [12]:
print(svd_data)

[[ 5.55005349e-03  1.63396438e-02 -1.19476090e-03 ...  4.19724207e-03
  -1.56562113e-02  9.63541812e-04]
 [-8.81915520e-13  2.27475554e-10  1.29251002e-10 ... -1.80633924e-04
   5.70759031e-04 -2.91509706e-04]
 [ 2.00698230e-03  7.27322918e-02 -2.78993579e-02 ... -3.25074830e-02
   5.53623643e-02  8.49914122e-03]
 ...
 [ 3.17235418e-13 -8.43683117e-11 -2.45299587e-10 ...  6.61987563e-04
  -9.68581215e-05 -3.77683911e-04]
 [ 1.17446705e-12  3.23930294e-10  7.94456779e-11 ...  1.03162300e-04
  -1.01392018e-03  3.51261124e-04]
 [ 9.56880669e-03  3.02958519e-02 -7.31958116e-04 ...  6.32142916e-02
  -8.79094858e-02  3.71458110e-02]]


<h2> Similaridade de Cosseno</h2>

<p>Uma vez que dois documentos são representados como vetores numéricos, é possível comparar a similaridade entre os documentos ao calcular o cosseno do ângulo entre esses documentos - uma medida de distância, não de amplitude. Para isso, basta resolver a equação do produto escalar entre os vetores, para encontrar o cosseno.</p>

<img src="http://s0.wp.com/latex.php?latex=++%5Cdisplaystyle++%5Cvec%7Ba%7D+%5Ccdot+%5Cvec%7Bb%7D+%3D+%5C%7C%5Cvec%7Ba%7D%5C%7C%5C%7C%5Cvec%7Bb%7D%5C%7C%5Ccos%7B%5Ctheta%7D+%5C%5C+%5C%5C++%5Ccos%7B%5Ctheta%7D+%3D+%5Cfrac%7B%5Cvec%7Ba%7D+%5Ccdot+%5Cvec%7Bb%7D%7D%7B%5C%7C%5Cvec%7Ba%7D%5C%7C%5C%7C%5Cvec%7Bb%7D%5C%7C%7D++&amp;bg=ffffff&amp;fg=000000&amp;s=0" alt="  \displaystyle  \vec{a} \cdot \vec{b} = \|\vec{a}\|\|\vec{b}\|\cos{\theta} \\ \\  \cos{\theta} = \frac{\vec{a} \cdot \vec{b}}{\|\vec{a}\|\|\vec{b}\|}  " title="  \displaystyle  \vec{a} \cdot \vec{b} = \|\vec{a}\|\|\vec{b}\|\cos{\theta} \\ \\  \cos{\theta} = \frac{\vec{a} \cdot \vec{b}}{\|\vec{a}\|\|\vec{b}\|}  " class="latex">

<img class=" wp-image-2582 " title="vector_space" src="http://blog.christianperone.com/wp-content/uploads/2013/09/vector_space.png" alt="" width="504" height="378">
<p fontsize=8>Fonte: http://blog.christianperone.com</p>

In [13]:
    documents = (
    "The sky is blue",
    "The sun is bright",
    "The sun in the sky is bright",
    "We can see the shining sun, the bright sun"
    )

In [19]:
documents

('The sky is blue',
 'The sun is bright',
 'The sun in the sky is bright',
 'We can see the shining sun, the bright sun')

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)
print(tfidf_matrix.shape)

(4, 11)


In [18]:
from sklearn.metrics.pairwise import cosine_similarity

query_vect = tfidf_vectorizer.transform(["The sun is red in the sky"])#ele segmenta e tokeniza automático, mas é bom criar o próprio

cosine_similarity(query_vect, tfidf_matrix)#próx de um é mais similar, é a terceira frase

array([[0.55245238, 0.59222761, 0.9467919 , 0.37986898]])

<p><b>Exercício 3:</b>Escreva um chatbot que, dado uma pergunta em Inglês, encontre uma pergunta mais parecida no corpus de perguntas e respostas disponível no Kaggle (https://www.kaggle.com/rtatman/questionanswer-dataset#S08_question_answer_pairs.txt) e exiba a resposta.